In [112]:
import pandas as pd 
import numpy as np
df_Assas = pd.read_csv('dados/asaas_export.csv',sep=';',decimal=',', dtype=str)
df_Manual = pd.read_csv('dados/controle_interno.csv',sep=';',decimal=',', dtype=str)

In [113]:
#DATATYPE em assas Celular, cpnj, Valortotal, datanegociacao 
#DATATYPE em manual valor,cnpj,data
#display(df_Assas.info())
display(df_Manual)

,Voucher,Nome da empresa,CNPJ,Data,Referencia,Iniciais,Situacao,Pagamento,Valor,Observacao,Nota enviada
0,13213531,HORIZONTE EDUCACAO LTDA,62777460000188,10/02/2026,Novo,Ds,Pago,Asaas,"210,00",retornar dia 28,NaN
1,13210953,VARANDA MARKETING LTDA,47.935.636/0001-99,01/01/2026,Google,Ds,Pago,Asaas,"210,00",aguardando nota,NaN
2,13206511,ESTEIO COMERCIO EIRELI,39128000000107,02/01/2026,Indicacao,Pc,Pago,Conta,"280,00",ok,NaN
3,13212453,VERTICE DISTRIBUIDORA S/A,52449915000182,17/02/2026,Renovacao,Ab,Pago,Conta,"190,00",ok,NaN
4,13201076,ERVAL TRANSPORTES LTDA,98.894.611/0001-25,17/02/2026,Apoio,So,Pago,Asaas,"210,00",NaN,sim
...,...,...,...,...,...,...,...,...,...,...,...
745,13202443,IPE EDUCACAO LTDA,57866725000184,03/01/2026,Renovacao,Bp,Em aberto,Conta,"190,00",contato por e-mail,sim
746,13214089,THIAGO BARROS,888.734.127-05,04/02/2026,Contador,Jp,Pago,Asaas,"210,00",contato por e-mail,sim
747,13200667,XISTO LOGISTICA LTDA,70787640000179,13/02/2026,Contrato,Ta,Em aberto,Asaas,"310,00",ok,NaN
748,13211532,NORTE ASSESSORIA LTDA,46.520.799/0001-48,13/01/2026,Contrato,Ds,Cortesia,Cortesia,"110,00",falar com o contador,NaN


In [114]:
#TRATAMENTO Manual Valores vazio de Manual arrumando datas Nota enviada

df_TratamentoManual = df_Manual.drop(columns=["Observacao"])
df_TratamentoManual ["Nota enviada"]= df_TratamentoManual["Nota enviada"].fillna("Enviar Nota")
df_TratamentoManual["Valor"] = df_TratamentoManual["Valor"].str.replace(",", ".")
df_TratamentoManual ["Data"]= pd.to_datetime(df_TratamentoManual["Data"], format='%d/%m/%Y')
df_TratamentoManual["Valor"]= pd.to_numeric(df_TratamentoManual["Valor"], errors="coerce" )
df_TratamentoManual["Nome da empresa"]= df_TratamentoManual["Nome da empresa"].str.strip()
df_TratamentoManual["Nome da empresa"]= df_TratamentoManual["Nome da empresa"].str.title()
vazio = df_TratamentoManual["Valor"].isna()
df_TratamentoManual["CNPJ"] = df_TratamentoManual["CNPJ"].str.replace(r"\D", "", regex=True)
print(pd.crosstab(df_TratamentoManual["Situacao"], vazio))
print("--------------")
print(pd.crosstab(df_TratamentoManual["Pagamento"], vazio))
df_TratamentoManual = df_TratamentoManual.rename(columns={
    "CNPJ":"CpfCnpj"
})








Valor      False  True 
Situacao               
Cortesia      48      1
Em aberto    120      2
Pago         570      9
--------------
Valor      False  True 
Pagamento              
Asaas        405      6
Conta        285      5
Cortesia      48      1


In [115]:
#Tratamento ASSAS
df_TratamentoAssas = df_Assas.rename(columns={
    "movimentacao|id": 'Id',
    "movimentacao|parceiro|nome": "Razao Social",
    "movimentacao|valorTotal": "ValorTotal",
    "movimentacao|dataNegociacao": "Data",
    "movimentacao|parceiro|cpfCnpj": "CpfCnpj",
    "movimentacao|parceiro|celular" : "Celular",
    "movimentacao|situacaoFinanceira": "SituacaoFinanceira",})
df_TratamentoAssas ["Data"]= pd.to_datetime(df_TratamentoAssas["Data"], format='%d/%m/%Y')
df_TratamentoAssas["ValorTotal"] = df_TratamentoAssas["ValorTotal"].str.strip()
df_TratamentoAssas["ValorTotal"] = df_TratamentoAssas["ValorTotal"].str.replace("R$ ", "")
df_TratamentoAssas["ValorTotal"] = df_TratamentoAssas["ValorTotal"].str.replace(",", ".")
df_TratamentoAssas["ValorTotal"] = pd.to_numeric(df_TratamentoAssas["ValorTotal"], errors="coerce")
df_TratamentoAssas["Razao Social"]= df_TratamentoAssas["Razao Social"].str.strip()
df_TratamentoAssas["Razao Social"]= df_TratamentoAssas["Razao Social"].str.title()

display(df_TratamentoAssas.nunique())


Id                    385
Razao Social          359
ValorTotal              8
Data                   60
Pago                    2
CpfCnpj               385
Celular               385
SituacaoFinanceira      2
dtype: int64

In [116]:
#separação de valores em branco que necessitam de revisão / percebi que teria que tratar os cnpj/cpf antes disso

df_TratamentoManual_Conferencia = df_TratamentoManual["CpfCnpj"].str.len().isin([11,14])
df_TratamentoManual_Revisao = df_TratamentoManual[~df_TratamentoManual_Conferencia]
df_TratamentoManual_Conferido = df_TratamentoManual[df_TratamentoManual_Conferencia] 
print(df_TratamentoManual["CpfCnpj"].str.len().value_counts())
print(df_TratamentoManual_Revisao.shape, df_TratamentoManual_Conferido.shape)

CpfCnpj
14.0    622
11.0    113
13.0      2
10.0      1
12.0      1
Name: count, dtype: int64
(15, 10) (735, 10)


In [117]:

df_AM_analise = df_TratamentoAssas.merge(
    df_TratamentoManual_Conferido, on="CpfCnpj",
    how="outer",
    indicator="Conferencia"
)

print(len(df_AM_analise))
print(df_AM_analise["Conferencia"].value_counts())


750
Conferencia
both          388
right_only    347
left_only      15
Name: count, dtype: int64


In [118]:
df_Divergencia_Manual = df_AM_analise[df_AM_analise["Conferencia"] == "right_only"]
print(df_Divergencia_Manual["Pagamento"].value_counts())

Pagamento
Conta       281
Cortesia     49
Asaas        17
Name: count, dtype: int64


In [119]:
Conferencias_final = [
    df_AM_analise["Conferencia"] == "left_only",
    (df_AM_analise["Conferencia"] == "right_only") & (df_AM_analise["Pagamento"] == "Asaas"),
    df_AM_analise["Conferencia"] == "right_only",
    (df_AM_analise["Conferencia"] == "both") & (df_AM_analise["Valor"].isna()),
    (df_AM_analise["Conferencia"] == "both") & (df_AM_analise["ValorTotal"] != df_AM_analise["Valor"]),
]

tipo = [
    "Cobranca_fora_do_manual",
    "Cobranca_nao_gerada",
    "Pago_fora_do_assas",
    "sem_valor_lancado",
    "valor_divergente",
]

df_AM_analise["Classificacao"] = np.select(Conferencias_final, tipo, default="Conferido")

print(df_AM_analise["Classificacao"].value_counts())

Classificacao
Conferido                  358
Pago_fora_do_assas         330
valor_divergente            24
Cobranca_nao_gerada         17
Cobranca_fora_do_manual     15
sem_valor_lancado            6
Name: count, dtype: int64
